# MarathonIQ — Expert Model Copy
### Date: 23/04/26 
### Description: copy of baseline - just with personal_best_minutes in feature set

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [2]:
# --- DATA LOADING ---
df_raw = pd.read_csv('train.csv')

print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")

Shape: (80000, 41)
Columns: ['runner_id', 'age', 'gender', 'running_experience_months', 'previous_marathon_count', 'training_program', 'motivation_level', 'personal_best_minutes', 'weekly_mileage_km', 'weekly_mileage_miles', 'runs_per_week', 'long_run_distance_km', 'speed_work_sessions_per_week', 'rest_days_per_week', 'training_adherence_pct', 'consecutive_weeks_no_miss', 'cross_training_hours_per_week', 'resting_heart_rate_bpm', 'vo2_max', 'bmi', 'recovery_score', 'sleep_hours_avg', 'injury_count', 'injury_severity', 'nutrition_score', 'hydration_consistency', 'training_streak_days', 'missed_workout_pct', 'early_morning_run_frequency', 'weather_condition_training_pct', 'goal_completion_rate', 'run_club_attendance_rate', 'warmup_adherence_pct', 'stretching_adherence_pct', 'marathon_date', 'marathon_weather', 'course_difficulty', 'target_finish_time_minutes', 'actual_finish_time_minutes', 'mental_preparation_score', 'medal_outcome']


In [3]:
df_raw['personal_best_minutes'].notna().sum()

69255

In [4]:
# --- ORDINAL ENCODING ---
df_clean = df_raw.copy()

training_map = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}
course_map   = {"Flat": 1, "Mixed": 2, "Hilly": 3}
injury_map   = {"Minor": 1, "Moderate": 2, "Severe": 3}

df_clean['training_program']  = df_clean['training_program'].map(training_map)
df_clean['course_difficulty'] = df_clean['course_difficulty'].map(course_map)
df_clean['injury_severity']   = df_clean['injury_severity'].map(injury_map).fillna(0)

In [5]:
# --- ONE-HOT ENCODING (nominal features) ---
ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe_array = ohe.fit_transform(df_clean[['gender', 'marathon_weather']])
ohe_df = pd.DataFrame(
    ohe_array,
    columns=ohe.get_feature_names_out(['gender', 'marathon_weather']),
    index=df_clean.index
)

df_clean = df_clean.drop(columns=['gender', 'marathon_weather'])
df_clean = pd.concat([df_clean, ohe_df], axis=1)

print(f"Shape after encoding: {df_clean.shape}")

Shape after encoding: (80000, 46)


In [6]:
# --- FEATURE DROPS ---
# Rationale documented per category

drop_list = [
    # Identifiers
    "runner_id",

    # Subjective 
    'recovery_score', 'injury_count', 'injury_severity', 'nutrition_score',
    

    # Leakage — derived from or identical to target => CHANGE: LEFT PB 
    "target_finish_time_minutes",
    "medal_outcome",


    # Redundant — duplicate information
    "weekly_mileage_miles",       # duplicate of weekly_mileage_km

    # Unparseable for baseline
    "marathon_date",

    # SHAP-informed drops — <0.5 min average impact on predictions (excl. age)
    "bmi",
    "consecutive_weeks_no_miss",
    "cross_training_hours_per_week",
    "early_morning_run_frequency",
    "gender_Male",
    "gender_Non-binary",
    "goal_completion_rate",
    "hydration_consistency",
    "long_run_distance_km",
    "marathon_weather_Sunny",
    "mental_preparation_score",
    "missed_workout_pct",
    "motivation_level",
    "previous_marathon_count",
    "rest_days_per_week",
    "runs_per_week",
    "sleep_hours_avg",
    "speed_work_sessions_per_week",
    "stretching_adherence_pct",
    "training_adherence_pct",
    "training_program",
    "training_streak_days",
    "warmup_adherence_pct",
    "weather_condition_training_pct",
]

df_clean = df_clean.drop(columns=[c for c in drop_list if c in df_clean.columns])
print(f"Shape after drops: {df_clean.shape}")
print(f"Remaining columns: {df_clean.columns.tolist()}")

Shape after drops: (80000, 13)
Remaining columns: ['age', 'running_experience_months', 'personal_best_minutes', 'weekly_mileage_km', 'resting_heart_rate_bpm', 'vo2_max', 'run_club_attendance_rate', 'course_difficulty', 'actual_finish_time_minutes', 'marathon_weather_Cold', 'marathon_weather_Hot', 'marathon_weather_Rainy', 'marathon_weather_Windy']


In [7]:
# --- REMAINING NaN features---
df_clean.isnull().sum()[df_clean.isnull().sum() > 0]

personal_best_minutes         10745
vo2_max                        9611
actual_finish_time_minutes     1989
dtype: int64

In [8]:
# --- FILTER DNFs & PREPARE MODEL DATA ---
df_model = df_clean[df_clean['actual_finish_time_minutes'].notna()].copy()

X = df_model.drop(columns=['actual_finish_time_minutes'])
y = df_model['actual_finish_time_minutes']

# Median imputation for remaining NaNs
X = X.fillna(X.median(numeric_only=True))

print(f"Model data shape: {df_model.shape}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Remaining NaNs: {X.isnull().sum().sum()}")

Model data shape: (78011, 13)
X shape: (78011, 12)
y shape: (78011,)
Remaining NaNs: 0


In [9]:
print(X.columns.tolist())

['age', 'running_experience_months', 'personal_best_minutes', 'weekly_mileage_km', 'resting_heart_rate_bpm', 'vo2_max', 'run_club_attendance_rate', 'course_difficulty', 'marathon_weather_Cold', 'marathon_weather_Hot', 'marathon_weather_Rainy', 'marathon_weather_Windy']


In [10]:
# --- TRAIN TEST SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (62408, 12)
X_test:  (15603, 12)


In [11]:
# --- SCALING ---
# Applied on the preprocessing file on all subsequent models. Note: X_train naming stays the same for simplicity.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train sucessfully scaled: {X_train.shape}")
print(f"X_test sucessfully scaled:  {X_test.shape}")

X_train sucessfully scaled: (62408, 12)
X_test sucessfully scaled:  (15603, 12)


In [12]:
# --- BASELINE: LINEAR REGRESSION ---
model_linreg = LinearRegression()

# Fit
model_linreg.fit(X_train, y_train)

# Predict
y_pred_linreg = model_linreg.predict(X_test)

# Evaluate
mae_linreg = mean_absolute_error(y_test, y_pred_linreg)
r2_linreg  = r2_score(y_test, y_pred_linreg)

print(f"LinReg MAE: {mae_linreg:.1f} minutes")
print(f"LinReg R2:  {r2_linreg:.3f}")

LinReg MAE: 13.2 minutes
LinReg R2:  0.738


In [13]:
# --- FINAL MODEL: XGBOOST ---
model_xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=6,
    random_state=42
)

# Fit
model_xgb.fit(X_train, y_train)

# Predict
y_pred_xgb = model_xgb.predict(X_test)

# Evaluate
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb  = r2_score(y_test, y_pred_xgb)

print(f"XGBoost MAE: {mae_xgb:.1f} minutes")
print(f"XGBoost R2:  {r2_xgb:.3f}")

XGBoost MAE: 12.0 minutes
XGBoost R2:  0.792


In [14]:
# --- RESULTS SUMMARY ---
results = pd.DataFrame({
    'Model':    ['LinearRegression', 'XGBoost'],
    'MAE (min)': [round(mae_linreg, 1), round(mae_xgb, 1)],
    'R2':        [round(r2_linreg, 3),  round(r2_xgb, 3)],
    'Features':  [X_train.shape[1],     X_train.shape[1]]
})

print(results.to_string(index=False))

           Model  MAE (min)    R2  Features
LinearRegression       13.2 0.738        12
         XGBoost       12.0 0.792        12
